In [2]:
from diffrax import diffeqsolve, ODETerm, SaveAt, Tsit5
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scienceplots

import Germany
from SIR import *

plt.style.use([
    'science',
    #'no-latex'
    ])
plt.rcParams['text.latex.preamble'] = r'\usepackage{newtxmath, newtxtext}'
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.Set1.colors)

In [3]:
# R0 = 12-18
# contagious period ~ 8 days
p = Parameters(alpha=1/8, beta=15/8, gamma=1 - 0.6**(1/365), W=Germany.OD_matrix, quarantine_factor=0.9)
term = ODETerm(f)
solver = Tsit5()

y0 = Germany.make_initial_condition()

In [4]:
t0 = 0.0
t1 = 100.0
dt0 = 1.0
saveat = SaveAt(ts=jnp.arange(t0, t1+dt0, dt0))

In [5]:
solution = diffeqsolve(term, solver, t0=t0, t1=t1, dt0=dt0, y0=y0, args=p, saveat=saveat)

In [6]:
ts = solution.ts
ys = solution.ys.reshape((-1, N_VARS, p.n_cities)) # type: ignore

In [7]:
# plot S, I, R reduced over counties
plt.figure()
S_reduced = jnp.sum(ys[:, 0, :], axis=1)
I_reduced = jnp.sum(ys[:, 1, :], axis=1)
R_reduced = jnp.sum(ys[:, 2, :], axis=1)
plt.plot(ts, S_reduced, label="S")
plt.plot(ts, I_reduced, label="I")
plt.plot(ts, R_reduced, label="R")
plt.xlabel("t [day]")
plt.legend()
plt.savefig("../figures/SIR.pdf")

In [8]:
# plot map of infected individuals day 0, 50, 100, ..., 300
for day in [0, 5, 10, 15, 20, 25, 30, 50, 100]:
    gdf = Germany.gdf.copy()
    plt.figure(figsize=(4,4))
    plt.title(f"Day {day}")
    S, I, R = ys[day, :, :]
    gdf['infected'] = I / (S + I + R)
    gdf.plot(column='infected', ax=plt.gca(), legend=True, cmap='Reds', edgecolor='black', linewidth=0.75, vmin=0, vmax=1)
    plt.show()

In [9]:
%matplotlib tk
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

fig, ax = plt.subplots(figsize=(8, 7))

norm = Normalize(vmin=0, vmax=1)
sm = ScalarMappable(norm=norm, cmap="Reds")
sm.set_array([])
fig.colorbar(sm, ax=ax, label="Infected fraction", shrink=0.7, pad=0.02)

def update(day):
    ax.clear()
    S, I, R = ys[day, :, :]
    gdf["infected"] = I / (S + I + R)
    gdf.plot(column="infected", ax=ax, cmap="Reds", edgecolor="black",
             linewidth=0.4, vmin=0, vmax=1)
    ax.set_title(f"Day {day}")
    ax.set_axis_off()

ani = FuncAnimation(fig, update, frames=range(len(ys)), interval=200)
plt.show()

In [10]:
%matplotlib tk
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# Calculate total S, I and R across counties
S_reduced = jnp.sum(ys[:, 0, :], axis=1)
I_reduced = jnp.sum(ys[:, 1, :], axis=1)
R_reduced = jnp.sum(ys[:, 2, :], axis=1)

fig, (ax_map, ax_sir) = plt.subplots(1, 2, figsize=(14, 7),
                                      gridspec_kw={"width_ratios": [1.2, 1]})
fig.subplots_adjust(wspace=0.25)

# Fixed colorbar
sm = ScalarMappable(norm=Normalize(vmin=0, vmax=1), cmap="Reds")
sm.set_array([])
fig.colorbar(sm, ax=ax_map, label="Infected fraction", shrink=0.7, pad=0.02)

# Set up SIR plot with fixed axes
ax_sir.set_xlim(float(ts[0]), float(ts[-1]))
ax_sir.set_ylim(0, float(max(jnp.max(S_reduced), jnp.max(I_reduced), jnp.max(R_reduced))) * 1.05)
ax_sir.set_xlabel("Time [days]")
ax_sir.set_ylabel("Number of people")
ax_sir.set_title("SIR model across all counties")
ax_sir.grid(alpha=0.3)

# Empty lines that will grow during the animation
line_S, = ax_sir.plot([], [], label="S")
line_I, = ax_sir.plot([], [], label="I")
line_R, = ax_sir.plot([], [], label="R")
ax_sir.legend()

def update(day):
    ax_map.clear()
    S, I, R = ys[day, :, :]
    gdf["infected"] = I / (S + I + R)
    gdf.plot(column="infected", ax=ax_map, cmap="Reds",
             edgecolor="black", linewidth=0.4, vmin=0, vmax=1)
    ax_map.set_title(f"Day {day}")
    ax_map.set_axis_off()

    # Reveal the curves up to the current timestep
    line_S.set_data(ts[:day+1], S_reduced[:day+1])
    line_I.set_data(ts[:day+1], I_reduced[:day+1])
    line_R.set_data(ts[:day+1], R_reduced[:day+1])

    return line_S, line_I, line_R

ani = FuncAnimation(
    fig, update,
    frames=range(0, len(ys), 1),  # last entry: how large gap between days
    interval=75,                 # ms between frames
    blit=False
)
plt.show()